# SLIIT Lecture Summarizer - Training Pipeline

**Approach:** Train a RandomForest classifier on SLIIT lecture PDFs to identify important sentences.

- **No external datasets** (no Kaggle, no arXiv)
- **Smart structural labeling** based on sentence patterns found in lecture content
- **TF-IDF vocabulary built from YOUR lectures** so it generalizes to any SLIIT module

---

## What This Notebook Does

1. Install dependencies
2. Mount Google Drive
3. Upload SLIIT lecture PDFs
4. Extract and clean sentences from PDFs
5. Label sentences using structural pattern matching
6. Train RandomForest model
7. Evaluate with human-verifiable test cases
8. Download model for local deployment

**Time:** ~10-15 minutes total

## STEP 1: Install Dependencies

In [ ]:
import subprocess, sys

packages = ['scikit-learn', 'nltk', 'pandas', 'numpy', 'PyPDF2', 'python-pptx']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
print('All dependencies installed.')

## STEP 2: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

WORK_DIR = '/content/drive/My Drive/SLIIT_Summarizer'
os.makedirs(f'{WORK_DIR}/training_data', exist_ok=True)
os.makedirs(f'{WORK_DIR}/models', exist_ok=True)
os.makedirs(f'{WORK_DIR}/lectures', exist_ok=True)
os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

## STEP 3: Upload Lecture PDFs

Upload **all** your SLIIT lecture PDFs. The more modules you upload, the better the model generalizes.

Minimum: 3-4 PDFs. Recommended: 6+ from different modules.

In [ ]:
from google.colab import files
import shutil

print('Upload your SLIIT lecture PDFs (select multiple files at once)...')
uploaded = files.upload()

for fname in uploaded.keys():
    shutil.move(fname, f'lectures/{fname}')
    print(f'  Saved: lectures/{fname}')

print(f'\nTotal files: {len(uploaded)}')

## STEP 4: Extract & Clean Sentences

In [ ]:
import re
import os
import pandas as pd
from PyPDF2 import PdfReader
from nltk.tokenize import sent_tokenize

# ── Lecturer name list (add more as needed) ─────────────────────────────────
LECTURER_NAMES = re.compile(
    r'\b[A-Z][a-z]+\s+[A-Z][a-z]*(arachchi|liyana|bandara|perera|silva|fernando|'
    r'wickrama|jayasena|dissanayake|rathnayake|kumara|senanayake|seneviratne|'
    r'rajapaksha|wijesinghe|gunasekara|pathirana|weerasinghe|lokuliyana|kasthuri|'
    r'anuradha|sliyanage|udara|sanvitha|shashika)\b',
    re.IGNORECASE
)

def clean_text(text):
    """
    Thorough cleaning of raw PDF text before sentence tokenisation.
    Removes: slide headers, lecturer names, URLs, code blocks,
             bullet symbols, unicode noise, and PDF artifacts.
    """
    # ── PDF extraction artifacts ──────────────────────────────────────────────
    text = re.sub(r'/g\d+', '', text)
    text = re.sub(r'\(cid:\d+\)', '', text)
    text = re.sub(r'\x0c', ' ', text)                         # form-feed
    text = re.sub(r'[\x00-\x08\x0b\x0e-\x1f\x7f]', '', text) # control chars

    # ── Unicode bullet / arrow / symbol characters ────────────────────────────
    text = re.sub(r'[•◼⚫●▪►▶✓✗✦✧◦‣⁃◘◙§¶†‡]', ' ', text)
    text = re.sub(r'[→←↑↓↔⇒⇐⇔]', ' ', text)
    text = re.sub(r'[ -⁯←-⇿─-◿]', ' ', text)

    # ── Slide header / footer patterns ───────────────────────────────────────
    text = re.sub(r'[A-Z]{2}\d{3,4}\s*\|[^.!?\n]*', ' ', text)
    text = re.sub(r'Module\s+Code\s*\|[^\n]*', ' ', text)
    text = re.sub(r'Lecture\s+Title\s*\|[^\n]*', ' ', text)
    text = re.sub(r'Faculty\s+of\s+Computing[^\n]*', ' ', text)
    text = re.sub(r'Department\s+of\s+(Information|Computer|Software)[^\n]*', ' ', text)
    text = re.sub(r'Sri Lanka Institute of Information Technology[^\n]*', ' ', text)
    text = re.sub(r'SLIIT\s*[–\-]\s*[^\n]*', ' ', text)

    # ── Lecturer names ────────────────────────────────────────────────────────
    text = LECTURER_NAMES.sub(' ', text)
    # Also catch "Dr. X" / "Prof. X" patterns
    text = re.sub(r'\b(Dr|Prof|Mr|Ms|Mrs)\.?\s+[A-Z][a-z]+\b', ' ', text)

    # ── URLs and web references ───────────────────────────────────────────────
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'www\.\S+', ' ', text)
    text = re.sub(r'Source\s*:\s*https?://[^\n]*', ' ', text)
    text = re.sub(r'Retrieved from[^\n]*', ' ', text)

    # ── Code-like content ─────────────────────────────────────────────────────
    # Remove lines that look like code (high punctuation density with braces/semicolons)
    lines = text.split('\n')
    clean_lines = []
    for line in lines:
        code_chars = sum(1 for c in line if c in '{}[]();=><')
        if len(line) > 10 and code_chars / max(len(line), 1) > 0.12:
            continue   # skip this line
        clean_lines.append(line)
    text = '\n'.join(clean_lines)

    # ── Slide numbering / assessment info ────────────────────────────────────
    text = re.sub(r'\b\d+\s*%\s*(Mid|End|Assignment|Lab|Practical|Test|Exam)\b', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\(Online\)|\(Written\)|\(Practical\)', ' ', text, flags=re.IGNORECASE)

    # ── Fix camelCase boundaries from slide extraction ────────────────────────
    text = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', text)

    # ── Normalise whitespace ──────────────────────────────────────────────────
    text = re.sub(r'\n{2,}', '. ', text)
    text = re.sub(r'(?<=[a-z])\n(?=[a-z])', ' ', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    text = text.strip()
    return text


def is_clean_sentence(s):
    """Return True if the sentence is worth keeping for training."""
    words = s.split()
    wc = len(words)

    if wc < 6 or wc > 80:
        return False

    # Must be mostly alphabetic
    alpha_ratio = sum(1 for c in s if c.isalpha()) / max(len(s), 1)
    if alpha_ratio < 0.55:
        return False

    # Reject jammed words (PDF extraction failure)
    if any(len(w) > 22 and not w.startswith('http') for w in words):
        return False

    # Reject residual bullet symbols
    if re.search(r'[•◼⚫●▪►▶]', s):
        return False

    # Reject residual lecturer names
    if LECTURER_NAMES.search(s):
        return False

    # Reject URLs
    if re.search(r'https?://|www\.', s):
        return False

    # Reject assessment/admin lines
    if re.search(r'\d+\s*hours?/(week|day)|attendance|enrolment key|contact (me|us)|reachable here', s, re.IGNORECASE):
        return False

    # Reject pure slide titles (all-caps with no sentence structure)
    upper_ratio = sum(1 for c in s if c.isupper()) / max(sum(1 for c in s if c.isalpha()), 1)
    if upper_ratio > 0.65 and wc < 10:
        return False

    return True


# ── Process all PDFs ──────────────────────────────────────────────────────────
all_sentences = []
lecture_files = sorted([f for f in os.listdir('lectures') if f.endswith('.pdf')])

print(f'Processing {len(lecture_files)} lecture PDFs...\n')

for fname in lecture_files:
    path = f'lectures/{fname}'
    raw_text = extract_text_from_pdf(path) if 'extract_text_from_pdf' in dir() else open(path,'rb').read().decode('utf-8', errors='ignore')

    # Use PyPDF2 for reading
    from PyPDF2 import PdfReader
    try:
        reader = PdfReader(path)
        raw_text = ''.join([(p.extract_text() or '') + ' ' for p in reader.pages])
    except Exception as e:
        print(f'  Error: {fname}: {e}')
        continue

    cleaned = clean_text(raw_text)
    raw_sentences = sent_tokenize(cleaned)

    kept = []
    for s in raw_sentences:
        s = s.strip()
        # Strip residual header fragments at sentence start
        s = re.sub(r'^[A-Z]{2}\d{3,4}[^.]*?(Lecture|Module)\s*\d*\s*', '', s).strip()
        s = re.sub(r'^\d+\s+', '', s).strip()
        s = re.sub(r'\s+\d+\s*$', '', s).strip()
        s = re.sub(r'\s+', ' ', s).strip()
        if is_clean_sentence(s):
            kept.append(s)

    for i, s in enumerate(kept):
        all_sentences.append({
            'sentence': s,
            'source_file': fname,
            'position_ratio': round(i / max(len(kept) - 1, 1), 4),
            'word_count': len(s.split())
        })

    print(f'  {fname}: {len(kept)} clean sentences (from ~{len(raw_sentences)} raw)')

df = pd.DataFrame(all_sentences)
print(f'\nTotal clean sentences: {len(df)}')
print(f'Per-file breakdown:')
print(df['source_file'].value_counts().to_string())
print(f'\nWord count stats:')
print(df['word_count'].describe().round(1).to_string())

df.to_csv('training_data/extracted_sentences.csv', index=False)
print(f'\nSaved to: training_data/extracted_sentences.csv')

## STEP 5: Label Sentences

This is the key step. Instead of a crude keyword counter, we use **structural pattern matching** that detects:

**Important (label=1):**
- Definitions: "X is defined as...", "X refers to...", "X is a..."
- Key concepts: sentences with technical terms, acronyms explained
- Formulas/rules: "The formula for...", normalized forms, constraints
- Enumerations: "There are N types of...", "The steps are..."
- Conclusions: "Therefore...", "In summary...", "The key point is..."

**Not important (label=0):**
- Slide headers/titles: "LECTURE 01 - INTRODUCTION"
- Instructor references: "Thank you!", email addresses
- Filler transitions: "As we discussed...", "Moving on..."
- Page/slide artifacts: "Slide 15 of 42", copyright notices
- Short fragments that lack substance

In [ ]:
import re
import numpy as np

# ── Business/soft-skills vocabulary (IT4070 false-positive suppression) ──────
_BUSINESS_WRITING_RE = re.compile(
    r'\b(cover letter|business letter|formal letter|informal letter|'
    r'memo|memorandum|circulars?|minutes|agenda|job application|'
    r'curriculum vitae|resume writing|'
    r'oral presentation|public speaking|presentation skills|'
    r'audience|speaker|listener|listening skills|body language|eye contact|'
    r'callers?|readers?|recipients?|addressee|'
    r'salutation|complimentary close|enclosure|subject line|'
    r'opening paragraph|closing paragraph|'
    r'report writing|executive summary|table of contents|'
    r'business communication|interpersonal (skills|communication)|'
    r'dress code|grooming|punctuality|etiquette|mannerism|'
    r'interviewing skills|interview tips|interview technique)\b',
    re.IGNORECASE
)

# ── Instructor names that sometimes survive cleaning ─────────────────────────
_INSTRUCTOR_NAMES_RE = re.compile(
    r'\b(sanvitha|kasthuriarachchi|anuradha|shashika|udara|lokuliyana|'
    r'weerasinghe|pathirana|gunasekara|wijesinghe|rajapaksha|seneviratne|'
    r'senanayake|kumara|rathnayake|dissanayake|jayasena|wickrama|fernando|'
    r'silva|perera|bandara|liyana)\b',
    re.IGNORECASE
)

# ── Technical vocabulary (CS/IT concepts) — used in soft scoring & functional_desc ──
_FUNC_TECH = re.compile(
    r'\b(data|system|network|protocol|server|client|query|table|process|thread|'
    r'memory|file|algorithm|function|class|object|database|schema|index|'
    r'transaction|security|architecture|layer|node|packet|buffer|cache|'
    r'stack|queue|tree|graph|cpu|os|disk|page|segment|register|bit|byte|'
    r'compiler|interpreter|kernel|scheduler|mutex|semaphore|deadlock|lock|'
    # Extended CS terms that must not fall out of the vocabulary
    r'primary key|foreign key|normalization|acid|atomicity|consistency|isolation|durability|'
    r'concurrency|replication|partition|encryption|authentication|authorization|'
    r'bandwidth|latency|throughput|subnet|routing|switching|firewall|'
    r'heap|pointer|recursion|polymorphism|inheritance|encapsulation|abstraction|'
    r'sorting|hashing|linked list|binary tree|relational|tuple|attribute|domain|'
    r'constraint|trigger|view|stored procedure|join|aggregate|subquery)\b',
    re.IGNORECASE
)


def label_sentence(sentence):
    """
    Label a sentence as important (1) or not (0) using structural patterns.
    Returns (label, confidence, reason).
    Designed to generalise across ANY SLIIT lecture module.
    """
    s = sentence.strip()
    sl = s.lower()
    words = s.split()
    word_count = len(words)

    # ====================================================================
    # HARD NEGATIVES  (label = 0, return immediately)
    # ====================================================================

    # 1. Slide headers / ALL-CAPS titles
    alpha_chars = [c for c in s if c.isalpha()]
    upper_ratio = sum(1 for c in alpha_chars if c.isupper()) / max(len(alpha_chars), 1)
    if upper_ratio > 0.70 and word_count < 12:
        return (0, 0.95, 'slide_header')

    # 2. Instructor / personal names
    if _INSTRUCTOR_NAMES_RE.search(s):
        return (0, 0.95, 'instructor_name')

    # 3. Email addresses
    if re.search(r'@\w+\.\w+', s):
        return (0, 0.95, 'email')

    # 4. Boilerplate
    if re.search(r'\b(copyright|all rights reserved|page \d+|slide \d+)\b', sl):
        return (0, 0.95, 'boilerplate')

    # 5. Closing / thank-you lines
    if re.search(r'\b(thank\s*you|acknowledgment|references?\s*$)\b', sl):
        return (0, 0.90, 'closing')

    # 6. Navigation / cross-reference lines
    if re.match(r'^(see|refer to|read|check|go through|look at)\s', sl):
        return (0, 0.88, 'reference')

    # 7. Filler transitions
    _FILLER = [
        r'^as (we |mentioned|discussed|noted|seen)',
        r'^(moving on|let us|let\'s|now we|we will now)',
        r'^in the (next|previous|last) (lecture|slide|section)',
        r'^you (already|should|can|will) (know|have|see)',
        r'^(note that this is just|this is just)',
        r'^(any questions?|questions? so far)',
        r'^below is (a|an|the)',    # ← catches "Below is a letter/example/draft..."
        r'^above is (a|an|the)',
        r'^here is (a|an|the)',
    ]
    for pat in _FILLER:
        if re.match(pat, sl):
            return (0, 0.82, 'filler_transition')

    # 8. Activity / exercise prompts
    if re.match(r'^(activity|exercise|homework|assignment|question)\s*[:,\-]', sl):
        return (0, 0.80, 'activity_prompt')

    # 9. IT4070 / business-writing content
    if _BUSINESS_WRITING_RE.search(sl):
        return (0, 0.82, 'business_writing')

    # 10. Too short with no technical signal
    if word_count < 7:
        if not re.search(r'[A-Z]{2,}|\d+[A-Z]|[a-z]+_[a-z]+', s):
            return (0, 0.80, 'too_short')

    # ====================================================================
    # HARD POSITIVES  (label = 1, return immediately)
    # ====================================================================

    # Subject must NOT be a personal pronoun (he/she/it/they/we/you/i/this+filler)
    _PRONOUN_START = re.compile(r'^(he|she|it|they|we|you|i)\s', re.IGNORECASE)

    # 11. Definitions — explicit definitional verb
    if not _PRONOUN_START.match(s):
        if re.match(
            r'^[A-Z][a-zA-Z\s\-/()\d]{0,45}?\b'
            r'(is defined as|is known as|refers to|can be defined as|means|is called)\b',
            s
        ) and word_count >= 8:
            return (1, 0.92, 'definition')

        # "Database is a collection…" / "Normalization is the process…"
        if re.match(
            r'^([A-Z][a-z]+(?:\s+[A-Za-z]+){0,3})\s+(is|are)\s+(a|an|the|used|considered|defined|known|responsible)\b',
            s
        ) and word_count >= 8:
            return (1, 0.90, 'definition')

        # "A database is…" / "An algorithm is…"
        if re.match(
            r'^(A|An|The)\s+[a-z][a-z\-]+(?:\s+[a-z]+){0,2}\s+(is|are)\s+(a|an|the|used|considered|defined)\b',
            s
        ) and word_count >= 8:
            return (1, 0.87, 'definition')

    # 12. Definition with dash/colon — requires acronym or title-case tech phrase
    _DASH_SEP = re.compile(r'^(.+?)\s[-–:]\s(.+)$')
    m = _DASH_SEP.match(s)
    if m and word_count >= 8:
        pre, post = m.group(1).strip(), m.group(2).strip()
        pre_words = pre.split()
        pre_is_acronym = re.match(r'^[A-Z]{2,}(\s[A-Z]{2,})*$', pre)
        pre_is_title_case = (
            len(pre_words) <= 4
            and all(w[0].isupper() for w in pre_words if w)
            and not re.search(r'\b(said|wrote|stated|argued|believed|noted)\b', sl)
            and not re.match(r'^(He|She|They|We|Below|Above|Here)\b', pre)
        )
        post_has_tech = bool(_FUNC_TECH.search(post.lower()))
        if (pre_is_acronym or pre_is_title_case) and post_has_tech:
            return (1, 0.85, 'definition_dash')

    # 13. Enumerations
    if re.search(
        r'there are (\d+|two|three|four|five|six|seven|eight|several|many|various)\s+'
        r'(types?|kinds?|categories|levels?|phases?|stages?|steps?|forms?|methods?|ways?|classes?|modes?)',
        sl
    ):
        return (1, 0.90, 'enumeration')

    # 14. Composition — includes? added
    if re.search(r'\b(consists? of|is composed of|comprises?|includes?|contain)\b', sl) and word_count >= 8:
        # guard against "includes" in non-technical context
        if _FUNC_TECH.search(sl) or re.search(r'\b[A-Z]{2,}\b', s):
            return (1, 0.85, 'composition')

    # 15. Formulas / rules / theorems (exclude English grammar rules)
    if re.search(r'\b(formula|equation|theorem|rule|law|axiom|constraint|condition)\b', sl) and word_count >= 7:
        if not re.search(r'\b(verb|noun|adjective|adverb|preposition|paragraph|grammar|spelling|tense|syllable|punctuation)\b', sl):
            return (1, 0.87, 'formula_rule')

    # 16. Named standards / protocols / forms
    if re.search(
        r'\b(\d+NF|BCNF|normal form|ACID|CAP theorem|OSI|TCP|UDP|HTTP|DNS|'
        r'DHCP|SNMP|SSL|TLS|OSPF|BGP|VLAN|VPN|RAID|FIFO|LRU|FCFS|SJF|'
        r'SQL|NoSQL|JDBC|ODBC|REST|SOAP|XML|JSON|HTML|CSS|IPv[46]|'
        r'deadlock|semaphore|mutex|primary key|foreign key|B-tree|'
        r'ACID properties|CAP|BASE|ETL|OLAP|OLTP|MapReduce|Hadoop|Spark)\b',
        s
    ) and word_count >= 7:
        return (1, 0.87, 'standard_protocol')

    # 17. Process descriptions
    if re.search(
        r'\b(the (process|procedure|algorithm|method|technique|approach|mechanism) (of|for|to|is|involves))\b',
        sl
    ):
        return (1, 0.85, 'process_desc')

    # 18. Advantages / disadvantages
    if re.search(
        r'\b(advantages?|disadvantages?|benefits?|drawbacks?|pros?|cons?|strengths?|weaknesses?|limitations?)\b',
        sl
    ) and word_count >= 7:
        return (1, 0.82, 'advantage_disadvantage')

    # 19. Conclusions / key takeaways
    if re.match(r'^(therefore|thus|hence|in summary|in conclusion|the key point|to summarize|consequently)\b', sl):
        return (1, 0.85, 'conclusion')

    # 20. Comparisons
    if re.search(r'\b(difference between|compared to|in contrast|unlike|whereas|while.*differs?)\b', sl) and word_count >= 8:
        return (1, 0.82, 'comparison')

    # 21. Acronym expansions  e.g. "SQL (Structured Query Language)"
    if re.search(r'[A-Z]{2,}\s*\([A-Z][a-z]', s) and word_count >= 6:
        return (1, 0.87, 'acronym_definition')

    # 22. Functional descriptions — requires a technical noun (tightened)
    if (
        re.search(r'\b(is used (to|for)|allows|enables|provides|ensures|guarantees|prevents|supports)\b', sl)
        and word_count >= 8
        and _FUNC_TECH.search(sl)
    ):
        return (1, 0.82, 'functional_desc')

    # ====================================================================
    # SOFT SCORING  (ambiguous sentences)
    # ====================================================================
    score = 0.0

    # Technical acronym density
    acronym_count = len(re.findall(r'\b[A-Z]{2,}\b', s))
    score += min(acronym_count * 0.12, 0.24)

    # Length sweet-spot
    if 10 <= word_count <= 50:
        score += 0.12
    elif word_count > 50:
        score -= 0.05

    # Core technical vocabulary (broader now)
    tech_hits = len(_FUNC_TECH.findall(sl))
    score += min(tech_hits * 0.07, 0.28)

    # Verb + object structure
    if re.search(r'\b(is|are|was|were|has|have|can|will|must|should|'
                 r'provides?|requires?|allows?|ensures?|creates?|stores?|manages?)\b', sl):
        score += 0.08

    # Causal / explanatory connectors
    if re.search(r'\b(because|since|due to|in order to|so that|as a result|leads to|causes|occurs when)\b', sl):
        score += 0.12

    # Illustrative examples
    if re.search(r'\b(for example|e\.g\.|such as|for instance)\b', sl):
        score += 0.08

    # Hedge / filler penalty
    if re.search(r'\b(obviously|clearly|of course|simply|just|basically)\b', sl):
        score -= 0.10

    # Business / soft-skills vocabulary penalty
    if re.search(r'\b(management|leadership|communication|teamwork|stakeholder|'
                 r'workplace|employee|employer|professional|career|resume|interview)\b', sl):
        score -= 0.10

    # ← lowered threshold from 0.35 to 0.28 to catch more genuine tech sentences
    if score >= 0.28:
        return (1, round(min(0.60 + score, 0.85), 2), 'soft_positive')
    elif score <= 0.08:
        return (0, 0.65, 'soft_negative')

    return (0, 0.55, 'ambiguous')


# ── Label all sentences ───────────────────────────────────────────────────────
print('Labeling sentences...\n')

labels, confidences, reasons = [], [], []
for _, row in df.iterrows():
    label, conf, reason = label_sentence(row['sentence'])
    labels.append(label)
    confidences.append(conf)
    reasons.append(reason)

df['importance']       = labels
df['label_confidence'] = confidences
df['label_reason']     = reasons

# ── Statistics ────────────────────────────────────────────────────────────────
n_pos = sum(labels)
n_neg = len(labels) - n_pos
print(f'Total sentences  : {len(df)}')
print(f'Important   (1)  : {n_pos} ({n_pos/len(df)*100:.1f}%)')
print(f'Not important (0): {n_neg} ({n_neg/len(df)*100:.1f}%)')

print(f'\nLabel-reason breakdown:')
print(df['label_reason'].value_counts().to_string())

print(f'\nPer-source-file positive rate:')
for src, grp in df.groupby('source_file'):
    pos = grp['importance'].sum()
    print(f'  {src:50s}  {pos}/{len(grp)} ({pos/len(grp)*100:.0f}% important)')

# Save
df.to_csv('training_data/labeled_sentences.csv', index=False)
print(f'\nSaved to: training_data/labeled_sentences.csv')


### STEP 5b: Verify Labels (sanity check)

Review samples from each label category to make sure the labeling is correct.

In [ ]:
print('=== SAMPLE IMPORTANT SENTENCES (label=1) ===\n')
pos_df = df[df['importance'] == 1].sample(min(10, n_pos), random_state=42)
for _, row in pos_df.iterrows():
    print(f'  [{row["label_reason"]:20s}] {row["sentence"][:120]}')

print(f'\n=== SAMPLE NOT-IMPORTANT SENTENCES (label=0) ===\n')
neg_df = df[df['importance'] == 0].sample(min(10, n_neg), random_state=42)
for _, row in neg_df.iterrows():
    print(f'  [{row["label_reason"]:20s}] {row["sentence"][:120]}')

print(f'\n=== HIGH-CONFIDENCE LABELS ===\n')
high_conf = df[df['label_confidence'] >= 0.85]
print(f'High-confidence labels (>=0.85): {len(high_conf)} ({len(high_conf)/len(df)*100:.1f}%)')
print(f'  Important: {len(high_conf[high_conf["importance"]==1])}')
print(f'  Not important: {len(high_conf[high_conf["importance"]==0])}')

## STEP 6: Train the Model

Features:
- **TF-IDF** (200 terms from lecture vocabulary, bigrams)
- **Structural features**: word count, avg word length, capitalization, digit ratio, punctuation
- **Pattern features**: has definition pattern, has acronym, has technical terms, sentence position

In [ ]:
import numpy as np
import pickle
import json
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report, confusion_matrix)

# ── Load labeled data ─────────────────────────────────────────────────────────
df_raw = pd.read_csv('training_data/labeled_sentences.csv')
df_raw = df_raw.dropna(subset=['sentence'])
df_raw['sentence'] = df_raw['sentence'].astype(str).str.strip()
df_raw = df_raw[df_raw['sentence'].str.len() > 10]
print(f'Raw labeled data: {len(df_raw)} sentences')
print(df_raw['source_file'].value_counts().to_string())

# ── Dataset balancing — cap each file's contribution ─────────────────────────
CAP_PER_FILE = 1200

balanced_parts = []
for src, grp in df_raw.groupby('source_file'):
    if len(grp) <= CAP_PER_FILE:
        balanced_parts.append(grp)
        print(f'  {src}: kept all {len(grp)} sentences')
    else:
        sampled = grp.sample(n=CAP_PER_FILE, random_state=42)
        balanced_parts.append(sampled)
        print(f'  {src}: capped {len(grp)} → {CAP_PER_FILE} sentences')

df = pd.concat(balanced_parts, ignore_index=True)
print(f'\nBalanced training set: {len(df)} sentences')
print(f'  Class 1 (important)    : {df["importance"].sum()}')
print(f'  Class 0 (not important): {len(df) - df["importance"].sum()}')

# ── Feature extraction — keep in sync with cell-10 _FUNC_TECH ────────────────
_FUNC_TECH_TRAIN = re.compile(
    r'\b(data|system|network|protocol|server|client|query|table|process|thread|'
    r'memory|file|algorithm|function|class|object|database|schema|index|'
    r'transaction|security|architecture|layer|node|packet|buffer|cache|'
    r'stack|queue|tree|graph|cpu|os|disk|page|segment|register|bit|byte|'
    r'compiler|interpreter|kernel|scheduler|mutex|semaphore|deadlock|lock|'
    r'primary key|foreign key|normalization|acid|atomicity|consistency|isolation|durability|'
    r'concurrency|replication|partition|encryption|authentication|authorization|'
    r'bandwidth|latency|throughput|subnet|routing|switching|firewall|'
    r'heap|pointer|recursion|polymorphism|inheritance|encapsulation|abstraction|'
    r'sorting|hashing|linked list|binary tree|relational|tuple|attribute|domain|'
    r'constraint|trigger|view|stored procedure|join|aggregate|subquery)\b',
    re.IGNORECASE
)

_PRONOUN_START_TRAIN = re.compile(r'^(he|she|it|they|we|you|i)\s', re.IGNORECASE)

def extract_structural_features(sentence, position_ratio=0.5):
    """
    21 non-TF-IDF features.  Must stay in sync with backend/app.py.
    """
    s = sentence
    sl = sentence.lower()
    words = s.split()
    word_count = len(words)

    def _is_definition(s):
        if _PRONOUN_START_TRAIN.match(s):
            return False
        if re.match(
            r'^[A-Z][a-zA-Z\s\-/()\d]{0,45}?\b'
            r'(is defined as|is known as|refers to|can be defined as|means|is called)\b', s
        ):
            return True
        if re.match(
            r'^([A-Z][a-z]+(?:\s+[A-Za-z]+){0,3})\s+(is|are)\s+(a|an|the|used|considered|defined|known|responsible)\b', s
        ):
            return True
        if re.match(
            r'^(A|An|The)\s+[a-z][a-z\-]+(?:\s+[a-z]+){0,2}\s+(is|are)\s+(a|an|the|used|considered|defined)\b', s
        ):
            return True
        return False

    features = [
        word_count,
        np.mean([len(w) for w in words]) if words else 0,
        sum(1 for c in s if c.isupper()) / max(len(s), 1),
        sum(1 for c in s if c.isdigit()) / max(len(s), 1),
        sum(1 for c in s if c in '.,;:!?') / max(len(s), 1),
        1 if _is_definition(s) else 0,
        1 if re.search(r'[A-Z]{2,}\s*\([A-Z][a-z]', s) else 0,
        1 if re.search(r'\b(consists? of|is composed of|comprises?|includes?|contain)\b', sl) and
             (_FUNC_TECH_TRAIN.search(sl) or re.search(r'\b[A-Z]{2,}\b', s)) else 0,
        1 if re.search(r'there are (\d+|two|three|four|five|six|several)', sl) else 0,
        1 if re.search(r'\b(therefore|thus|hence|in summary|in conclusion)\b', sl) else 0,
        1 if re.search(r'\b(advantage|disadvantage|benefit|drawback|limitation)\b', sl) else 0,
        1 if re.search(r'\b(because|since|due to|in order to|so that)\b', sl) else 0,
        1 if re.search(r'\b(for example|e\.g\.|such as|for instance)\b', sl) else 0,
        1 if re.search(r'\b(difference between|compared to|in contrast|unlike)\b', sl) else 0,
        1 if (re.search(r'\b(is used (to|for)|allows|enables|provides|ensures)\b', sl)
              and _FUNC_TECH_TRAIN.search(sl)) else 0,
        len(re.findall(r'\b[A-Z]{2,}\b', s)),
        len(_FUNC_TECH_TRAIN.findall(sl)),
        position_ratio,
        1 if re.search(r'@\w+\.\w+', s) else 0,
        1 if re.search(r'copyright|all rights reserved', sl) else 0,
        1 if re.match(r'^(see|refer to|read|check)\s', sl) else 0,
    ]
    return features

STRUCTURAL_FEATURE_NAMES = [
    'word_count', 'avg_word_len', 'capital_ratio', 'digit_ratio', 'punct_ratio',
    'is_definition', 'has_acronym_def', 'is_composition', 'is_enumeration',
    'is_conclusion', 'is_pros_cons', 'is_causal', 'is_example', 'is_comparison',
    'is_functional', 'acronym_count', 'tech_term_count', 'position_ratio',
    'has_email', 'is_boilerplate', 'is_reference'
]

# ── Build TF-IDF vocabulary ───────────────────────────────────────────────────
print('\nBuilding TF-IDF vocabulary from lecture content...')

_NAME_STOPWORDS = [
    'sanvitha', 'kasthuriarachchi', 'anuradha', 'shashika', 'udara',
]
_sklearn_stop = list(TfidfVectorizer(stop_words='english').get_stop_words())
_combined_stop = _sklearn_stop + _NAME_STOPWORDS

vectorizer = TfidfVectorizer(
    max_features=200,
    min_df=2,           # ← was 3, lowered to 2 so rare-but-critical CS terms survive
    max_df=0.85,
    ngram_range=(1, 2),
    stop_words=_combined_stop
)

tfidf_matrix = vectorizer.fit_transform(df['sentence'].values).toarray()
print(f'TF-IDF vocabulary: {len(vectorizer.vocabulary_)} terms')

# Spot-check that critical CS terms made it in
cs_check = ['acid', 'deadlock', 'primary', 'foreign', 'normalization', 'semaphore']
found = [t for t in cs_check if any(t in k for k in vectorizer.vocabulary_)]
missing = [t for t in cs_check if not any(t in k for k in vectorizer.vocabulary_)]
print(f'CS key terms present in vocab: {found}')
if missing:
    print(f'Still missing (too rare in corpus): {missing}')

# ── Extract structural features ───────────────────────────────────────────────
print('\nExtracting structural features...')
structural_features = np.array([
    extract_structural_features(row['sentence'], row.get('position_ratio', 0.5))
    for _, row in df.iterrows()
])

X = np.hstack([tfidf_matrix, structural_features])
y = df['importance'].values

print(f'\nFinal feature matrix: {X.shape}')
print(f'  TF-IDF features   : {tfidf_matrix.shape[1]}')
print(f'  Structural features: {structural_features.shape[1]}')

# ── Train / test split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain: {len(X_train)}, Test: {len(X_test)}')

# ── Train RandomForest ────────────────────────────────────────────────────────
print('\nTraining RandomForest...')
n_neg_tr = (y_train == 0).sum()
n_pos_tr = (y_train == 1).sum()
weight_ratio = n_neg_tr / max(n_pos_tr, 1)
print(f'Class weight ratio (neg/pos): {weight_ratio:.2f}x')

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight={0: 1, 1: max(2, int(weight_ratio))},
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
print('Training complete.')

# ── Evaluate ──────────────────────────────────────────────────────────────────
y_pred    = model.predict(X_test)
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)

print(f'\n{"=" * 50}')
print(f'MODEL PERFORMANCE')
print(f'{"=" * 50}')
print(f'Accuracy:  {accuracy:.4f}  ({accuracy*100:.1f}%)')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1 Score:  {f1:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Not Important', 'Important']))
cm = confusion_matrix(y_test, y_pred)
print(f'Confusion Matrix:')
print(f'  TN={cm[0][0]}  FP={cm[0][1]}')
print(f'  FN={cm[1][0]}  TP={cm[1][1]}')

print(f'\n5-Fold Cross-Validation...')
cv_scores = cross_val_score(model, X, y, cv=5, scoring='f1')
print(f'  CV F1 scores: {[f"{s:.3f}" for s in cv_scores]}')
print(f'  Mean: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})')

from sklearn.metrics import roc_curve
from sklearn.metrics import f1_score as f1_fn

probs_test = model.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, probs_test)

best_thresh, best_f1 = 0.5, 0.0
for t in thresholds:
    f = f1_fn(y_test, (probs_test >= t).astype(int))
    if f > best_f1:
        best_f1, best_thresh = f, t

print(f'\nOptimal threshold: {best_thresh:.3f}')
print(f'Tuned F1:          {best_f1:.4f}')
y_pred_tuned = (probs_test >= best_thresh).astype(int)
print('\nTuned Classification Report:')
print(classification_report(y_test, y_pred_tuned, target_names=['Not Important', 'Important']))


### STEP 6b: Feature Importance Analysis

In [ ]:
# Feature importance analysis
importances = model.feature_importances_
n_tfidf = tfidf_matrix.shape[1]

tfidf_importance = importances[:n_tfidf].sum()
structural_importance = importances[n_tfidf:].sum()

print(f'Feature importance distribution:')
print(f'  TF-IDF features:    {tfidf_importance:.3f} ({tfidf_importance*100:.1f}%)')
print(f'  Structural features: {structural_importance:.3f} ({structural_importance*100:.1f}%)')

print(f'\nTop 10 structural features:')
struct_importances = importances[n_tfidf:]
struct_sorted = sorted(zip(STRUCTURAL_FEATURE_NAMES, struct_importances), key=lambda x: x[1], reverse=True)
for name, imp in struct_sorted[:10]:
    print(f'  {name:25s} {imp:.4f}')

print(f'\nTop 10 TF-IDF terms:')
tfidf_importances = importances[:n_tfidf]
vocab_items = sorted(vectorizer.vocabulary_.items(), key=lambda x: x[1])
tfidf_sorted = sorted(zip([v[0] for v in vocab_items], tfidf_importances), key=lambda x: x[1], reverse=True)
for term, imp in tfidf_sorted[:10]:
    print(f'  {term:25s} {imp:.4f}')

## STEP 7: Test with Real Sentences

Verify the model produces sensible predictions on known examples.

In [ ]:
test_cases = [
    # Should be IMPORTANT
    ('A database is a collection of related data organized for efficient access and retrieval.', 1),
    ('Normalization is the process of organizing data to reduce redundancy and improve integrity.', 1),
    ('The primary key uniquely identifies each record in a relational database table.', 1),
    ('SQL (Structured Query Language) is a standard language for managing relational databases.', 1),
    ('There are three levels of database architecture: external, conceptual, and internal.', 1),
    ('ACID properties include Atomicity, Consistency, Isolation, and Durability.', 1),
    ('A foreign key is a field that refers to the primary key of another table.', 1),
    ('The process scheduling algorithm determines which process runs next on the CPU.', 1),
    ('TCP provides reliable, ordered delivery of data between applications.', 1),
    ('Deadlock occurs when two or more processes are waiting for each other to release resources.', 1),

    # Should be NOT IMPORTANT
    ('Thank you for attending this lecture.', 0),
    ('See chapter 5 for more details on this topic.', 0),
    ('As we discussed in the previous lecture, we will continue.', 0),
    ('Slide 15 of 42 page break continued.', 0),
    ('Copyright 2024 All Rights Reserved SLIIT.', 0),
    ('Moving on to the next section of our discussion.', 0),
    ('Please refer to the textbook for additional reading.', 0),
    ('You already know these concepts from the previous module.', 0),
    ('Any questions so far about what we covered?', 0),
    ('Let us now look at another example.', 0),
]

print(f'{"=" * 60}')
print('HUMAN-VERIFIABLE TEST CASES')
print(f'{"=" * 60}\n')

correct = 0
total = len(test_cases)

for sentence, expected in test_cases:
    tfidf_feat = vectorizer.transform([sentence]).toarray()
    struct_feat = np.array([extract_structural_features(sentence, 0.5)])
    features = np.hstack([tfidf_feat, struct_feat])

    pred = model.predict(features)[0]
    prob = model.predict_proba(features)[0][1]

    match = 'OK' if pred == expected else 'WRONG'
    if pred == expected:
        correct += 1

    label = 'IMP' if pred == 1 else 'NOT'
    exp_label = 'IMP' if expected == 1 else 'NOT'
    print(f'  [{match:5s}] pred={label} exp={exp_label} prob={prob:.2f} | {sentence[:75]}')

print(f'\nTest accuracy: {correct}/{total} ({correct/total*100:.0f}%)')

## STEP 8: Save & Download Model

In [ ]:
import pickle
import json

# Save model
with open('models/universal_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Saved: models/universal_model.pkl')

# Save vectorizer
with open('models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
print('Saved: models/vectorizer.pkl')

# Save metadata
metadata = {
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1': float(f1),
    'cv_f1_mean': float(cv_scores.mean()),
    'cv_f1_std': float(cv_scores.std()),
    'optimal_threshold': float(best_thresh),
    'training_approach': 'RandomForest on SLIIT lecture PDFs with structural labeling',
    'training_data': f'{len(df)} sentences from {len(df["source_file"].unique())} lecture PDFs',
    'n_features': int(X.shape[1]),
    'n_tfidf_features': int(tfidf_matrix.shape[1]),
    'n_structural_features': int(structural_features.shape[1]),
    'class_distribution': {
        'important': int(df['importance'].sum()),
        'not_important': int(len(df) - df['importance'].sum())
    }
}

with open('models/universal_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('Saved: models/universal_metadata.json')

print(f'\n{json.dumps(metadata, indent=2)}')


In [ ]:
from google.colab import files

print('Downloading model files...\n')
print('Copy these 3 files to your local project backend/ folder:')
print('  1. universal_model.pkl')
print('  2. vectorizer.pkl')
print('  3. universal_metadata.json')
print()

files.download('models/universal_model.pkl')
files.download('models/vectorizer.pkl')
files.download('models/universal_metadata.json')

print('\nDone! Files also saved in Google Drive: SLIIT_Summarizer/models/')